In [31]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Embedding

In [5]:
reviews = ['nice food',
          'amazing restaurant',
          'too good',
          'just loved it',
          'will go again',
          'horrible food',
          'never go home',
          'poor service',
          'poor quality',
          'needs improvement']
sentiment = np.array([1,1,1,1,1,0,0,0,0,0])

In [6]:
one_hot("amazing restaurant",30)

[24, 21]

In [35]:
vocab_size = 30
tokenizer = Tokenizer(num_words=vocab_size,oov_token="<OOV>")
tokenizer.fit_on_texts(reviews)

In [36]:
word_index = tokenizer.word_index
word_index

{'<OOV>': 1,
 'food': 2,
 'go': 3,
 'poor': 4,
 'nice': 5,
 'amazing': 6,
 'restaurant': 7,
 'too': 8,
 'good': 9,
 'just': 10,
 'loved': 11,
 'it': 12,
 'will': 13,
 'again': 14,
 'horrible': 15,
 'never': 16,
 'home': 17,
 'service': 18,
 'quality': 19,
 'needs': 20,
 'improvement': 21}

In [37]:
encoded_reviews = tokenizer.texts_to_sequences(reviews)
encoded_reviews

[[5, 2],
 [6, 7],
 [8, 9],
 [10, 11, 12],
 [13, 3, 14],
 [15, 2],
 [16, 3, 17],
 [4, 18],
 [4, 19],
 [20, 21]]

In [38]:
lengths = [len(d) for d in encoded_reviews]
max_length = max(lengths)
max_length

3

In [39]:
padded_reviews = pad_sequences(encoded_reviews,max_length,padding="post")
padded_reviews

array([[ 5,  2,  0],
       [ 6,  7,  0],
       [ 8,  9,  0],
       [10, 11, 12],
       [13,  3, 14],
       [15,  2,  0],
       [16,  3, 17],
       [ 4, 18,  0],
       [ 4, 19,  0],
       [20, 21,  0]])

In [41]:
embedded_vector_size = 4

X = padded_reviews
y = sentiment

model = Sequential()
model.add(Embedding(vocab_size,embedded_vector_size,input_length=max_length,name="embedding"))
model.add(Flatten())
model.add(Dense(1,activation="sigmoid"))

model.compile(
    optimizer = "adam",
    loss = "binary_crossentropy",
    metrics = ["accuracy"]
)
model.fit(X,y,epochs=20,verbose=0)
model.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ (None, 3, 4)                │             120 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_8 (Flatten)                  │ (None, 12)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_8 (Dense)                      │ (None, 1)                   │              13 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 401 (1.57 KB)

 Trainable params: 133 (532.00 B)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 268 (1.05 KB)

In [42]:
model.evaluate(X,y)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step - accuracy: 1.0000 - loss: 0.6727


[0.6727265119552612, 1.0]

In [43]:
weights = model.get_layer("embedding").get_weights()[0]
len(weights)

30

In [50]:
def getVector(word):
    index = word_index.get(word)
    if index<vocab_size and index is not None:
        print(f"'{word}' => {weights[index]}")
    else:
        print(f"'{word}' not found or outside vocab limit.")

getVector("nice")
getVector("good")

'nice' => [-0.02804298 -0.05068577  0.02518938  0.04898892]
'good' => [-0.05374354 -0.016468    0.01326073 -0.03291183]
